In [1]:
from torch import nn
import torch


In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weights = nn.Parameter(torch.ones(dim))
    def forward(self, X):
        var_X = x.pow(2).mean(dim=1, keepdim=True)
        x_normed = X * torch.sqrt(var_X + self.eps)
        return x_normed * self.weights

In [3]:
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.w_gate = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_up = nn.Linear(d_model, ffn_dim, bias=False)
        self.w_down = nn.Linear(ffn_dim, d_model, bias=False)
    def forward(self, X):
        gate = F.silu(self.w_gate(X))
        up = self.w_up(X)
        down = self.w_down(gate * up)
        return down
        

In [17]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.num_queries_per_kv = num_heads // num_kv_heads # Коэффициент повтора GQA (g)

        self.q_proj = nn.Linear(d_model, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(num_heads * self.head_dim, d_model, bias=False)
        
    def forward(self, X, kv_cache=None):
        # X -> [B, T_q, D]
        B, T_q, D = X.shape
        q = self.q_proj(X).view(B, T_q, self.num_heads, self.head_dim).transpose(1, 2)
        k_new = self.k_proj(X).view(B, T_q, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v_new = self.v_proj(X).view(B, T_q, self.num_kv_heads, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k_new], dim=2)
            v = torhc.cat([v_prev, v_new], dim=2)
        else:
            k, v = k_new, v_new
        new_kv_cache = (k, v)
        if self.num_queries_per_kv > 1:
            # [B, H_kv, T_total, d_h] -> [B, H, T_total, d_h]
            k_exec = torch.repeat_interleave(k, dim=1, repeats=self.num_queries_per_kv)
            v_exec = torch.repeat_interleave(v, dim=1, repeats=self.num_queries_per_kv)
        else:
            k_exec, v_exec = k, v

        scores = torch.matmul(q, k_exec.transpose(-1, -2)) / math.sqrt(self.head_dim)
        T_total = k.size(2)
        T_past = T_total - T_q
        if T_q > 1:
            causal_mask = torch.triu(torch.full((T_q, T_total), float('-inf'), device=X.device),
                diagonal=T_past + 1)
            scores = scores + causal_mask.unsqueeze(0).unsqueeze(0)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v_exec).transpose(1, 2).contiguous().view(B, T_q, -1)
        out = self.out_proj(context)
        return out, new_kv_cache

In [19]:
B, T_q, D = 2, 8, 256
num_heads = 8
num_kv_heads = 2

attn = CausalSelfAttention(d_model=D, num_heads=num_heads, num_kv_heads=num_kv_heads)
x = torch.randn(B, T_q, D)